# NOVA SN68 — scoring BoltzGen de nanobodies sur GPU Kaggle

Note des séquences avec **la version de BoltzGen embarquée par les validateurs NOVA**, avec leur structure cible (Q9NZQ7 / 8AOK), leur site de liaison, leur protocole et leurs échelles.

Notebook **autonome** : la source BoltzGen, la structure, le lot (`inputs/kaggle_boltzgen_timing.csv`) et le code de scoring sont intégrés. La MSA est téléchargée via jsDelivr et vérifiée par empreinte SHA-256 ; les poids (~6 Go) viennent de Hugging Face.

**Réglages Kaggle :** *Accelerator* **GPU T4 x2**, *Internet* **On**.
- Lot complet (8 séquences, ~0.2 h à 1,5 min/séquence (mesuré sur 84) sur 2×T4) : **Save Version → Save & Run All (Commit)**.
- Test rapide : `TOP = 2` puis **Run All** dans l'éditeur.

Le lot alterne candidats et séquences de calibration, donc même un run interrompu compare les deux.

Les noyaux cuEquivariance se désactivent d'eux-mêmes sur T4 (capacité 7.5 < 8). Mesuré : **3,7 min/séquence** sur 2×T4 (design 162 s, repliement 213 s, analyse 23 s pour 2 séquences).

In [ ]:
# Paramètres
TOP = None          # None = tout le lot ; 2 pour un test rapide
RUN = "boltzgen_timing"
REPLICATE = 1       # incrémenter pour une réplique : BoltzGen est stochastique, mais il
                    # REPREND les résultats déjà présents, donc un nom de run neuf est
                    # indispensable pour obtenir un vrai second tirage.

import base64, io, os, pathlib, subprocess, sys, tarfile, time
T0 = time.time()
TEMP = pathlib.Path("/kaggle/temp")
WORK = TEMP / "Nova"
NOVA = TEMP / "nova"
VENV = TEMP / "bgvenv"
OUT = pathlib.Path("/kaggle/working")

RUN = RUN if REPLICATE == 1 else f"{RUN}_rep{REPLICATE}"

def step(msg):
    print(f"[{time.strftime('%H:%M:%S')}] +{(time.time() - T0) / 60:5.1f} min  {msg}", flush=True)

def sh(cmd, check=True, **kw):
    # a string goes through the shell, a list does not
    res = subprocess.run(cmd, shell=isinstance(cmd, str), text=True, capture_output=True, **kw)
    out = (res.stdout + res.stderr).strip()
    if out:
        print(out[-3000:])
    if check and res.returncode != 0:
        raise RuntimeError(f"échec ({res.returncode}) : {cmd}")
    return res.stdout

step("démarrage")
# Un notebook réimporté repart avec l'accélérateur par défaut : sans GPU, BoltzGen
# serait inutilisable, autant le dire tout de suite.
import shutil
if shutil.which("nvidia-smi") is None:
    raise SystemExit("Aucun GPU : mettre 'Accelerator' sur 'GPU T4 x2' dans les "
                     "paramètres du notebook (panneau de droite), puis relancer.")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
TEMP.mkdir(parents=True, exist_ok=True)
sh("df -h /kaggle/temp | tail -1", check=False)

In [ ]:
# Données intégrées : BoltzGen des validateurs, structure 8AOK, inputs/kaggle_boltzgen_timing.csv, code de scoring (tar.xz base64)
MSA_NAME = "Q9NZQ7.a3m"
MSA_SHA256 = "0bd1fc43143c7881b52a35cc42903eb3ea37c4791e17f5f3837c20b59b5101c0"
MSA_URLS = ["https://cdn.jsdelivr.net/gh/metanova-labs/nova@db9bf190a6907338a05871b9658f42a089229675/data/msa_files/Q9NZQ7.a3m", "https://raw.githubusercontent.com/metanova-labs/nova/db9bf190a6907338a05871b9658f42a089229675/data/msa_files/Q9NZQ7.a3m"]
PAYLOAD = "..."

In [ ]:
step("test de l'accès Internet")
for url in ["https://huggingface.co", "https://pypi.org", "https://cdn.jsdelivr.net"]:
    rc = subprocess.run(["curl", "-s", "-o", "/dev/null", "-m", "15", "-w", "%{http_code}", url],
                        capture_output=True, text=True).stdout or "000"
    print(f"  {url}: {rc}")
    if rc == "000":
        raise SystemExit("Pas d'accès Internet : activer 'Internet' dans les paramètres du notebook.")

In [ ]:
step("extraction du code, de la source BoltzGen et de la structure")
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:xz") as tar:
    tar.extractall(WORK)
(TEMP / "nova").mkdir(parents=True, exist_ok=True)
sh(f"cp -r {WORK}/nova/. {NOVA}/ && rm -rf {WORK}/nova")
step("téléchargement de la MSA des validateurs")
import hashlib
msa = NOVA / "data/msa_files" / MSA_NAME
msa.parent.mkdir(parents=True, exist_ok=True)
for url in MSA_URLS:
    subprocess.run(["curl", "-sSL", "-m", "180", "--retry", "2", "-o", str(msa), url])
    if msa.exists() and hashlib.sha256(msa.read_bytes()).hexdigest() == MSA_SHA256:
        print(f"  MSA OK depuis {url}")
        break
    print(f"  échec depuis {url}")
else:
    raise RuntimeError("MSA introuvable ou corrompue")
sh(f"find {NOVA} -maxdepth 3 -type d | sort && ls {WORK}")
step("étape 1 terminée")

## Environnement BoltzGen
La version des validateurs épingle `numpy==2.0.2`, `torch==2.7.1` et une trentaine d'autres paquets : elle est installée dans un venv séparé (compter ~10 min).

In [ ]:
step("installation de uv + Python 3.12")
sh(f"{sys.executable} -m pip install -q uv")
sh(f"uv venv -q --python 3.12 {VENV}")
py = str(VENV / "bin/python")
step("installation de torch 2.7.1 CUDA")
sh(f"uv pip install -q --python {py} torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126")
step("installation de la BoltzGen des validateurs (+ pandas, requests)")
sh(f"uv pip install -q --python {py} {NOVA}/boltzgen pandas requests")
# La BoltzGen de NOVA importe bittensor pour neuf appels de journalisation ; un
# substitut minimal evite d'installer ce paquet lourd dans l'environnement epingle.
site = pathlib.Path(sh([py, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"]).strip())
sh(["cp", str(WORK / "kaggle" / "bittensor.py"), str(site / "bittensor.py")])
sh(f'{py} -c "import torch, boltzgen, bittensor; print(torch.__version__, torch.cuda.is_available(), '
   f'torch.cuda.get_device_capability(0))"')
step("étape 2 terminée")

## Scoring
Le premier lancement télécharge les poids BoltzGen (~6 Go). La progression s'affiche toutes les 5 minutes avec la dernière ligne du journal ; le journal complet est dans `/kaggle/working/boltzgen_<RUN>.log`.

In [ ]:
import pandas as pd
BATCH = "inputs/kaggle_boltzgen_timing.csv"
os.chdir(WORK)
n_total = len(pd.read_csv(BATCH).head(TOP) if TOP else pd.read_csv(BATCH))
log_path = OUT / f"boltzgen_{RUN}.log"
cmd = [py, "-m", "scripts.boltzgen_score", "--file", BATCH, "--run", RUN,
       "--nova-dir", str(NOVA), "--boltzgen", str(VENV / "bin/boltzgen")]
if TOP:
    cmd += ["--top", str(TOP)]

def last_log_line():
    lines = [l.strip() for l in log_path.read_text(errors="replace").replace("\r", "\n").splitlines() if l.strip()]
    return lines[-1][:200] if lines else "(journal vide)"

out_dir = WORK / f"data/boltzgen/{RUN}/out"
if out_dir.exists():
    raise SystemExit(f"{out_dir} existe déjà : BoltzGen reprendrait les résultats au lieu de "
                     f"recalculer. Incrémenter REPLICATE pour un nouveau tirage.")
step(f"lancement de BoltzGen sur {n_total} séquences (run {RUN})")
# Chronometrage de l etage BoltzGen SEUL, sans l installation ni les 6 Go de
# poids : c est le seul chiffre du budget d epoque qui manque, parce que le
# conteneur ou tourne le mineur n a pas de GPU. Tous les autres etages sont
# mesures (FINDINGS 7bis).
t_boltzgen = time.time()
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    while proc.poll() is None:
        time.sleep(300)
        step(f"en cours — {last_log_line()}")
elapsed = time.time() - t_boltzgen

print(log_path.read_text(errors="replace").replace("\r", "\n")[-5000:])
step(f"BoltzGen terminé (code {proc.returncode})")
assert proc.returncode == 0, "BoltzGen a échoué : voir le journal ci-dessus"
print()
print("=== CHRONOMETRAGE DE L ETAGE BOLTZGEN ===")
print(f"  {n_total} sequences en {elapsed / 60:.1f} min "
      f"= {elapsed / max(n_total, 1):.0f} s par sequence")
try:
    import torch
    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    print(f"  sur {len(gpus)} GPU : {', '.join(gpus)}")
except Exception as exc:
    print(f"  (GPU non identifies : {exc})")
print(f"  budget d epoque : 67 min au total, dont ~20 min deja mesurees hors")
print(f"  BoltzGen, donc marge restante {67 - 20 - elapsed / 60:.0f} min")

In [ ]:
import shutil
src = WORK / f"data/boltzgen/{RUN}/scores.csv"
out = OUT / f"scores_nanobody_{RUN}.csv"
shutil.copy(src, out)
df = pd.read_csv(out)
step(f"résultats : {out} ({len(df)} séquences)")
df